In [ ]:
# 기본 라이브러리
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

from PIL import Image

print("환경 설정 완료")

환경 설정 완료


In [ ]:
import zipfile
import os

zip_path = "/content/archive.zip"
extract_path = "/content/stanford_cars"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("압축 해제 완료")
print(os.listdir(extract_path))

BadZipFile: File is not a zip file

라벨 읽기

In [ ]:
# 데이터 기본 경로 설정
base_path = "/content/stanford_cars"

# 실제 데이터 폴더 연결
train_img_path = os.path.join(base_path, "cars_train")
test_img_path = os.path.join(base_path, "cars_test")
devkit_path = os.path.join(base_path, "car_devkit")

print("데이터 경로:", base_path)
print("Train 이미지 폴더:", os.path.exists(train_img_path))
print("Test 이미지 폴더:", os.path.exists(test_img_path))
print("Devkit 폴더:", os.path.exists(devkit_path))

print(os.listdir(devkit_path))

In [ ]:
# 실제 devkit 경로 수정
devkit_path = os.path.join(
    base_path,
    "car_devkit",
    "devkit"
)

print("Devkit 경로:", devkit_path)
print(os.listdir(devkit_path))

클래스 이름 읽기

In [ ]:
meta_path = os.path.join(
    devkit_path,
    "cars_meta.mat"
)
train_anno_path = os.path.join(devkit_path, "cars_train_annos.mat")
meta_mat = sio.loadmat(meta_path)

print(meta_mat.keys())

In [ ]:
class_names_raw = meta_mat["class_names"][0]

class_names = [
    name[0] if isinstance(name[0], str) else str(name[0])
    for name in class_names_raw
]

print("클래스 수:", len(class_names))
print(class_names[:10])


(각 차종이 몇 장씩 있는지 실제 데이터를 기준으로 확인한다.)

Train annotation에서 클래스 번호를 추출

In [ ]:
train_mat = sio.loadmat(train_anno_path)
annotations = train_mat["annotations"][0]


train_labels = []

for ann in annotations:
    label = int(ann["class"][0][0])
    train_labels.append(label)

print("Train 이미지 수:", len(train_labels))
print("클래스 종류:", len(set(train_labels)))

196개 클래스 분포표 만들기

In [ ]:
class_counts = pd.Series(train_labels).value_counts().sort_index()

distribution_df = pd.DataFrame({
    "class_id": range(1, 197),
    "class_name": class_names,
    "image_count": [
        class_counts.get(i, 0)
        for i in range(1, 197)
    ]
})

print("전체 클래스 수:", len(distribution_df))
print("전체 Train 이미지 수:", distribution_df["image_count"].sum())

display(distribution_df)

최소한 통계도 같이 확인

In [ ]:
print("최소 이미지 수:", distribution_df["image_count"].min())
print("최대 이미지 수:", distribution_df["image_count"].max())
print("평균 이미지 수:", distribution_df["image_count"].mean())
print("중앙값:", distribution_df["image_count"].median())

그리고 많은 클래스부터:

In [ ]:
distribution_df.sort_values(
    "image_count",
    ascending=False
).head(20)

적은 클래스도:

In [ ]:
distribution_df.sort_values(
    "image_count"
).head(20)

이미지 샘플 확인

Train 이미지 폴더:

In [ ]:
train_img_path = os.path.join(base_path, "cars_train")

print("이미지 개수:", len(os.listdir(train_img_path)))

현재 분포에서 확인된 것

196개 클래스가 모두 존재하고, Train 총 8,144장인데 클래스당 이미지 수가 대략 32~46장 수준이야.

즉 Stanford Cars는:

클래스 수: 196
Train: 8,144장
클래스당 평균: 약 41.6장
클래스별 수량은 크게 치우치지 않고 비교적 균등
하지만 클래스당 데이터 자체는 상당히 적음

이번에는 196개 중 일부 클래스만 임의로 보는 게 아니라, 전체를 어느 정도 대표할 수 있도록 12개 클래스를 골라서 실제 사진을 확인



In [ ]:
print(os.listdir(train_img_path)[:20])

In [ ]:
print("cars_train 내부:")
!find /content/stanford_cars/cars_train -maxdepth 2 -type f | head -20

In [ ]:
# 실제 이미지가 들어있는 폴더로 경로 수정
train_img_path = os.path.join(
    base_path,
    "cars_train",
    "cars_train"
)

print("실제 Train 이미지 경로:", train_img_path)
print("이미지 파일 수:", len(os.listdir(train_img_path)))

In [ ]:
# ⑤ 실제 이미지 샘플 확인
# 196개 클래스를 고르게 살펴보기 위해 12개 클래스를 선택

sample_class_ids = np.linspace(1, 196, 12, dtype=int)

plt.figure(figsize=(18, 12))

for plot_idx, class_id in enumerate(sample_class_ids):

    # 해당 클래스의 annotation 찾기
    class_annotations = [
        ann for ann in annotations
        if int(ann["class"][0][0]) == class_id
    ]

    # 해당 클래스에서 첫 번째 이미지
    ann = class_annotations[0]

    filename = ann["fname"][0]
    image_path = os.path.join(train_img_path, filename)

    image = Image.open(image_path)

    plt.subplot(3, 4, plot_idx + 1)
    plt.imshow(image)
    plt.title(
        f"Class {class_id}\n{class_names[class_id - 1]}",
        fontsize=10
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

한 클래스 안에서 이미지가 얼마나 다양한지

In [ ]:
# 한 클래스의 여러 이미지 확인
# 첫 번째 클래스(AM General Hummer SUV 2000)를 예시로 확인

target_class_id = 1

class_annotations = [
    ann for ann in annotations
    if int(ann["class"][0][0]) == target_class_id
]

plt.figure(figsize=(16, 10))

for i, ann in enumerate(class_annotations[:12]):

    filename = ann["fname"][0]
    image_path = os.path.join(train_img_path, filename)

    image = Image.open(image_path)

    plt.subplot(3, 4, i + 1)
    plt.imshow(image)
    plt.axis("off")

plt.suptitle(
    f"Class {target_class_id}: {class_names[target_class_id - 1]}",
    fontsize=16
)

plt.tight_layout()
plt.show()

"클래스 내부에서도 사진이 충분히 다양한가?"4개 클래스 × 8장 = 32장

In [ ]:
# 클래스 내부 이미지 다양성 확인
# 대표적으로 4개 클래스를 선택해서 각 8장씩 확인

target_class_ids = [1, 36, 107, 196]

plt.figure(figsize=(16, 16))

plot_idx = 1

for class_id in target_class_ids:

    class_annotations = [
        ann for ann in annotations
        if int(ann["class"][0][0]) == class_id
    ]

    for ann in class_annotations[:8]:

        filename = ann["fname"][0]
        image_path = os.path.join(train_img_path, filename)

        image = Image.open(image_path)

        plt.subplot(4, 8, plot_idx)
        plt.imshow(image)
        plt.axis("off")

        plot_idx += 1

plt.tight_layout()
plt.show()

실제 이미지에서 확인된 특징

Class 1 Hummer

정면/사선/측면 등 촬영 각도가 다양함
배경도 실외, 자연, 흰 배경 등 다양함
화질 차이도 있음

Class 36 BMW M6 Convertible

정면·측면·후면 등 다양한 각도
실내 전시장, 도로, 스튜디오 이미지 등이 섞여 있음
일부 이미지에는 광고성 텍스트가 존재

Class 107 Ford Mustang Convertible

전면/측면/사선 등 다양한 각도
주차장, 딜러샵, 도로 등 배경이 다양함
같은 차종이어도 이미지 스타일이 상당히 다름
이건 좋은 관찰이야

따라서 현재 샘플 기준으로는 "각 클래스가 거의 동일한 사진만 가지고 있다"는 문제는 보이지 않아.

반면 다음 문제는 확인됐어.

웹 수집 이미지 특성상 배경·광고 문구·이미지 품질 등의 노이즈가 존재한다.

이건 나중에 Data Audit → 오류 분석 → 모델이 차량 자체가 아닌 배경/텍스트에 의존할 가능성을 확인하는 근거로 쓸 수 있어.

지금까지 데이터 자체는 실제 자동차 사진 + 다양한 환경 + 클래스 균등 분포라는 점에서 꽤 괜찮아 보여. 다만 클래스당 32~46장이라는 작은 표본 문제와 광고/배경 편향 가능성은 반드시 포트폴리오에서 다뤄야 해.

라벨과 실제 이미지 연결

여기서 우리가 볼 것

단순히 "자동차 사진이네"만 보는 게 아니야.

다음 5가지를 확인할 거야.

차량이 실제 사진에서 충분히 크게 보이는가
배경이 다양한가
차량 각도가 다양한가
차량 일부가 가려진 사진이 있는가
클래스 간 외형 차이가 실제로 존재하는가

특히 마지막이 중요해.

예를 들어

Sedan A vs Sedan B

처럼 비슷하게 생긴 차종들이 섞여 있다면 CNN + 오류 분석 프로젝트에 상당히 좋은 조건이 될 수 있어.

반대로 특정 차종이

특정 배경 + 특정 촬영각도 + 특정 로고

같은 것으로 거의 구분된다면 background/shortcut bias 문제를 따로 확인해야 해.

전체 무결성 검사

In [ ]:
# ⑥ 데이터 무결성 검사
# 파일 존재 여부 + 이미지 열기 가능 여부 + 이미지 기본 정보

import os
from PIL import Image

missing_files = []
broken_files = []
image_sizes = []
image_modes = []

for i, ann in enumerate(annotations):

    filename = ann["fname"][0]
    image_path = os.path.join(train_img_path, filename)

    # 1. 파일 존재 확인
    if not os.path.exists(image_path):
        missing_files.append(filename)
        continue

    # 2. 이미지 정상 열림 확인
    try:
        with Image.open(image_path) as img:
            img.verify()

        # 기본 정보 다시 확인
        with Image.open(image_path) as img:
            image_sizes.append(img.size)
            image_modes.append(img.mode)

    except Exception:
        broken_files.append(filename)

print("===== 데이터 무결성 검사 =====")
print("Annotation 수:", len(annotations))
print("실제 이미지 수:", len(os.listdir(train_img_path)))
print("파일 없음:", len(missing_files))
print("이미지 손상/읽기 오류:", len(broken_files))

print("\n===== 이미지 기본 정보 =====")
print("서로 다른 이미지 크기 수:", len(set(image_sizes)))
print("주요 이미지 모드:", set(image_modes))

print("\n===== 결과 =====")

if len(missing_files) == 0 and len(broken_files) == 0:
    print("✅ 모든 Train annotation이 실제 이미지와 정상적으로 연결됨")
    print("✅ 깨진 이미지 없음")
else:
    print("⚠️ 확인이 필요한 파일이 있음")

L/RGB가 클래스별로 어떻게 분포하는지

In [ ]:
# RGB / Grayscale 이미지가 특정 클래스에 몰려 있는지 확인

mode_by_class = []

for ann in annotations:
    filename = ann["fname"][0]
    class_id = int(ann["class"][0][0])
    image_path = os.path.join(train_img_path, filename)

    with Image.open(image_path) as img:
        mode = img.mode

    mode_by_class.append({
        "class_id": class_id,
        "class_name": class_names[class_id - 1],
        "mode": mode
    })

mode_df = pd.DataFrame(mode_by_class)

mode_table = pd.crosstab(
    mode_df["class_name"],
    mode_df["mode"]
)

display(mode_table)

전체 비율

In [ ]:
print(mode_df["mode"].value_counts())
print("\n비율:")
print(mode_df["mode"].value_counts(normalize=True).round(3))

"일부러 클래스별로 그레이스케일과 컬러를 섞어 놓은 데이터" 문제는 사실상 아니라고 봐도 돼.

결과가 아주 명확해
전체: 8,144장
RGB: 8,126장 (99.8%)
Grayscale(L): 18장 (0.2%)
196개 클래스 중 대부분이 RGB만 존재
특정 클래스가 L 이미지로 몰려 있는 패턴도 보이지 않음

즉 18장의 L 이미지는 극소수 예외 데이터야.

클레스 선정하기

In [ ]:
# ⑥ 클래스 선정 후보 1차 확인
# 이미지 수가 많은 순서가 아니라,
# 각 클래스의 데이터 수를 한눈에 확인

candidate_df = distribution_df.copy()

print("이미지 수 분포")
print(candidate_df["image_count"].describe())

print("\n이미지 수 구간별 클래스 수")
print(pd.cut(
    candidate_df["image_count"],
    bins=[0, 34, 39, 44, 50],
    include_lowest=True
).value_counts().sort_index())

클래스별 이미지 수 분포 그래프

In [ ]:
plt.figure(figsize=(18, 6))

plt.bar(
    candidate_df["class_id"],
    candidate_df["image_count"]
)

plt.xlabel("Class ID")
plt.ylabel("Number of training images")
plt.title("Stanford Cars - Training Images per Class")

plt.tight_layout()
plt.show()

클래스 선정 1차 후보 추리기

In [ ]:
# ⑥ 클래스 선정 - 1차 후보군 만들기
# 차체 유형별로 클래스 확인

body_types = [
    "Sedan",
    "SUV",
    "Convertible",
    "Coupe",
    "Hatchback",
    "Van",
    "Wagon",
    "Pickup",
    "Minivan"
]

for body_type in body_types:
    print(f"\n{'='*60}")
    print(f"[{body_type}]")
    print(f"{'='*60}")

    matched = distribution_df[
        distribution_df["class_name"].str.contains(
            body_type,
            case=False,
            na=False
        )
    ]

    print(f"클래스 수: {len(matched)}")

    for _, row in matched.iterrows():
        print(
            f"{int(row['class_id']):3d} | "
            f"{row['class_name']} | "
            f"{int(row['image_count'])}장"
        )

제조사가 동일해서 차량 세부 특징을 비교하기 좋음.같은 제조사 안에서 모델/연식이 비슷한 차종이 있어서 fine-grained 분류와 오류 분석이 가능함.

In [ ]:
# ⑥ 클래스 선정 - Audi / BMW / Hyundai 후보 이미지 확인

candidate_pairs = {
    "Audi": [23, 24],        # S4 Sedan 2012 / S4 Sedan 2007
    "BMW": [32, 33],         # X5 SUV 2007 / X6 SUV 2012
    "Hyundai": [134, 138]    # Sonata Hybrid / Sonata Sedan
}

for manufacturer, class_ids in candidate_pairs.items():

    print(f"\n{'='*70}")
    print(f"{manufacturer}")
    print(f"{'='*70}")

    plt.figure(figsize=(16, 5))

    plot_idx = 1

    for class_id in class_ids:

        class_annotations = [
            ann for ann in annotations
            if int(ann["class"][0][0]) == class_id
        ]

        for ann in class_annotations[:4]:

            filename = ann["fname"][0]
            image_path = os.path.join(train_img_path, filename)

            image = Image.open(image_path).convert("RGB")

            plt.subplot(2, 4, plot_idx)
            plt.imshow(image)
            plt.title(
                f"Class {class_id}\n{class_names[class_id - 1]}",
                fontsize=10
            )
            plt.axis("off")

            plot_idx += 1

    plt.tight_layout()
    plt.show()

In [ ]:
# ⑦ 최종 후보 6개 데이터 현황 확인

selected_classes = [23, 24, 32, 33, 134, 138]

selected_df = distribution_df[
    distribution_df["class_id"].isin(selected_classes)
].copy()

selected_df = selected_df.sort_values("class_id")

print("===== 최종 후보 클래스 =====")
display(selected_df)

print("\n===== Train 데이터 합계 =====")
print("클래스 수:", len(selected_df))
print("이미지 수:", selected_df["image_count"].sum())

In [ ]:
# ⑧ 선택한 6개 클래스의 공식 Test 데이터 확인

test_anno_path = os.path.join(
    devkit_path,
    "cars_test_annos.mat"
)

test_mat = sio.loadmat(test_anno_path)
test_annotations = test_mat["annotations"][0]

test_labels = []

for ann in test_annotations:
    label = int(ann["class"][0][0])
    test_labels.append(label)

test_counts = pd.Series(test_labels).value_counts().sort_index()

selected_test_df = pd.DataFrame({
    "class_id": selected_classes,
    "class_name": [
        class_names[i - 1]
        for i in selected_classes
    ],
    "test_image_count": [
        test_counts.get(i, 0)
        for i in selected_classes
    ]
})

print("===== 선택한 6개 클래스의 공식 Test 데이터 =====")
display(selected_test_df)

print("\n공식 Test 전체 이미지:", len(test_annotations))
print("선택 클래스 Test 이미지:", selected_test_df["test_image_count"].sum())

In [ ]:
# Test annotation 실제 구조 확인

print(test_mat.keys())

print("\nannotations 구조:")
print(test_mat["annotations"].dtype)
print(test_mat["annotations"][0].dtype.names)

적은양의 데이터셋 라벨링없는 test데이터로 추가 6개 클레스 선정

In [ ]:
from collections import Counter
import pandas as pd

# 현재 선택한 6개 클래스 번호
selected_classes = [23, 24, 32, 33, 134, 138]

# 클래스별 이미지 수
label_counts = Counter(train_labels)

# 후보 데이터프레임
candidate_data = []

for class_id in range(1, 197):
    count = label_counts[class_id]

    # 현재 선택 클래스 제외
    if class_id in selected_classes:
        continue

    # 너무 적은 클래스 제외
    if count < 35:
        continue

    candidate_data.append({
        "class_id": class_id,
        "class_name": class_names[class_id - 1],
        "train_images": count
    })

candidates = pd.DataFrame(candidate_data)

# 이미지 수가 많은 순서
candidates = candidates.sort_values(
    by="train_images",
    ascending=False
).reset_index(drop=True)

print(f"추가 후보 클래스 수: {len(candidates)}")
display(candidates.head(50))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for name in dirs + files:
        if 'stanford' in name.lower() or 'cars' in name.lower():
            print(os.path.join(root, name))

In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for name in files:
        if name.lower().endswith('.zip'):
            print(os.path.join(root, name))